In [ ]:
import pandas as pd
import json

from aloud_database.aloud_database import Database

db = Database()

query = """
    SELECT
        c.conversion_raw_info
    FROM
        lead_tracking_prod.conversions c 
    WHERE
        c.conversion_type_id = '2'
"""


df = db.execute_query(query=query)
# Supondo que seu DataFrame seja df e a coluna seja 'conversion_raw_info'

#df = df[df['campaign_id'] == 'BF24']

# Se a coluna vier como string JSON, converter para dict
df["conversion_raw_info"] = df["conversion_raw_info"].apply(
    lambda x: json.loads(x) if isinstance(x, str) else x
)

# Função recursiva para extrair todas as chaves de um JSON
def extract_keys(obj, parent_key=""):
    keys = set()
    if isinstance(obj, dict):
        for k, v in obj.items():
            full_key = f"{parent_key}.{k}" if parent_key else k
            keys.add(full_key)
            keys |= extract_keys(v, full_key)
    elif isinstance(obj, list):
        for i, item in enumerate(obj):
            keys |= extract_keys(item, parent_key)
    return keys

# Aplicar em todas as linhas
all_keys = set()
for row in df["conversion_raw_info"].dropna():
    all_keys |= extract_keys(row)

# Converter para lista ordenada
all_keys_list = sorted(all_keys)

# Exibir resultado
for k in all_keys_list:
    print(k)

In [ ]:
# Lista de variações possíveis
income_keys = [
    "monthly_income",
    "monthly_incomme",
    "monthly_personal_income",
    "monthy_income"
]

# Função para extrair o valor de income do JSON
def get_monthly_income(row):
    if not isinstance(row, dict):
        return None
    for k in income_keys:
        if k in row:
            return row[k]
    return None

# Criando as novas colunas
df["monthly_incomme"] = df["conversion_raw_info"].apply(get_monthly_income)